# Classification d'images à l'aide d'algorithmes de Deep Learning

Projet n&#8239;$^\text{o}$ 6 du [cursus Machine Learning Engineer][2] d'OpenClassrooms

Auteur : [Kiril ISAKOV][1]

Mentor : Nicolas TISSERAND

Projet démarré le 23/03/2026

[1]: https://github.com/kirisakow/
[2]: https://openclassrooms.com/fr/paths/794-machine-learning-engineer

## Prérequis : installer les pilotes GPU, etc.

1. Installer ou mettre à jour le pilote pour la GPU. Par ici pour les GPU Nvidia : https://www.nvidia.com/en-us/drivers/ (spécifier le bon modèle dans le formulaire)

2. Installer le dernier CUDA toolkit (v13.2) de chez Nvidia : 

   - soit de la façon [`runfile (local)`][2] :

     ```bash
     # télécharger l'archive (taille approximative : 4,1 Go)
     wget https://developer.download.nvidia.com/compute/cuda/13.2.0/local_installers/cuda_13.2.0_595.45.04_linux.run

     # exécuter l'archive
     sudo sh cuda_13.2.0_595.45.04_linux.run
     ```

   - soit de la façon [`deb (network)`][3] :

     ```bash
     wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-keyring_1.1-1_all.deb
     
     sudo dpkg -i cuda-keyring_1.1-1_all.deb
     
     sudo apt update
     
     sudo apt -y install cuda-toolkit-13-2
     ```

3. Ajouter les chemins suivants aux variables d'environnement PATH et LD_LIBRARY_PATH :

     ```bash
     echo 'export PATH=/usr/local/cuda/bin:$PATH' >> ~/.bashrc
     echo 'export LD_LIBRARY_PATH=/usr/local/cuda/lib64:/usr/lib/x86_64-linux-gnu:$LD_LIBRARY_PATH' >> ~/.bashrc

     # Enfin, recharger .bashrc
     source ~/.bashrc
     ```

4. Installer le backend pour Keras (PyTorch, TensorFlow ou JAX) :

   - Pour PyTorch : Suivre [les instructions de la doc PyTorch][1] : Installer le backend `torch` and `torchvision` compatibles CUDA `>=13.0` pour les GPU Nvidia (ou ROCm `>=7.2` pour les GPU AMD).

   - Si besoin, spécifier dans `pyproject.toml` des bornes plus strictes à la version de Python : par exemple, non pas `>=3.12` mais `>=3.12,<3.14`.

5. Installer Keras : Suivre [les instructions de la doc Keras][4]

[1]: https://pytorch.org/get-started/locally/
[2]: https://developer.nvidia.com/cuda-downloads?target_os=Linux&target_arch=x86_64&Distribution=Ubuntu&target_version=24.04&target_type=runfile_local
[3]: https://developer.nvidia.com/cuda-downloads?target_os=Linux&target_arch=x86_64&Distribution=Ubuntu&target_version=24.04&target_type=deb_network
[4]: https://keras.io/getting_started/

# Notebook de création et d’entraînement du modèle personnel

## Imports et fonctions

In [1]:
from functions_img_preprocessing import (
    apply_gaussian_blur,
    convert_to_grayscale,
    crop_image,
    equalize_histogram,
    get_boundingbox,
    get_breed,
    mirror_image,
    normalize_image,
    resize_image,
    whiten_image,
)
import os
os.environ["KERAS_BACKEND"] = "torch" # valoriser cette variable d'environnement avant d'importer keras
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import keras
from keras import layers
from keras.preprocessing import image_dataset_from_directory

from collections import Counter
from lxml import etree
from pathlib import Path
from PIL import Image
from pprint import pprint
from sklearn.model_selection import train_test_split
import cv2
import itertools as it
import numpy as np

import logging
logging.getLogger('tensorflow').setLevel(logging.WARNING)
import warnings
warnings.filterwarnings("ignore")

## Étapes préliminaires de test de l'installation

### Tester l'installation de CUDA toolkit et de PyTorch

In [2]:
display(
    f"PyTorch version: {torch.__version__}",
    f"Is CUDA available: {torch.cuda.is_available()}",
    f"CUDA device count: {torch.cuda.device_count()}",
    f"Current device: {torch.cuda.current_device()}",
    f"Device name: {torch.cuda.get_device_name(0)}",
)

'PyTorch version: 2.11.0+cu130'

'Is CUDA available: True'

'CUDA device count: 1'

'Current device: 0'

'Device name: NVIDIA GeForce RTX 2070 with Max-Q Design'

## Entraînement du modèle de prédiction *from scratch*

Recenser tous les répertoires contenant les images et les annotations :

In [4]:
img_file_paths = tuple(Path('images/').glob('*/'))
annot_file_paths = tuple(Path('annotations/').glob('*/'))

img_file_paths[0]

PosixPath('images/n02105251-briard')

Les répertoires contenant les images, triés en fonction du nombre d'images :

In [5]:
bag_of_file_counts = {
    dir_path: sum(1 for f in dir_path.iterdir() if f.is_file())
    for dir_path in img_file_paths
}
bag_of_file_counts_sorted_desc = dict(sorted(bag_of_file_counts.items(), reverse=True, key=lambda item: item[1]))
display(
    "Les répertoires contenant le plus d'images :",
    dict(list(bag_of_file_counts_sorted_desc.items())[:5]),
    "Les répertoires contenant le moins d'images :",
    dict(list(bag_of_file_counts_sorted_desc.items())[-5:]),
)

"Les répertoires contenant le plus d'images :"

{PosixPath('images/n02085936-Maltese_dog'): 252,
 PosixPath('images/n02088094-Afghan_hound'): 239,
 PosixPath('images/n02092002-Scottish_deerhound'): 232,
 PosixPath('images/n02112018-Pomeranian'): 219,
 PosixPath('images/n02090721-Irish_wolfhound'): 218}

"Les répertoires contenant le moins d'images :"

{PosixPath('images/n02115913-dhole'): 150,
 PosixPath('images/n02106166-Border_collie'): 150,
 PosixPath('images/n02101556-clumber'): 150,
 PosixPath('images/n02086079-Pekinese'): 149,
 PosixPath('images/n02090379-redbone'): 148}

### Expérience 1 : Premier entraînement à 3 classes, d'abord sans *data augmentation*, ensuite avec

Nous allons donc sélectionner trois races selon le nombre d'images mises à notre disposition :

- Deux races contenant le plus d'images : **`Maltese_dog`** (252 images) et **`Afghan_hound`** (239 images).
- Une race contenant le moins d'images : **`redbone`** (148 images).

Split train - val - test :

In [6]:
input_img_paths = tuple(list(bag_of_file_counts_sorted_desc.keys())[i] for i in [-1, 0, 1])
display(input_img_paths)
input_img_paths = tuple(path.glob('*.jpg') for path in input_img_paths)
input_img_paths = tuple(it.chain.from_iterable(input_img_paths))
print(f"Résultat : un tuple de {len(input_img_paths)} images pour les 3 classes de races.")
breed_labels = tuple(str(path).split('/')[1] for path in input_img_paths)
print("Distribution des races :", )
pprint(dict(Counter(breed_labels)))
train_frac = 0.8
input_paths_train, input_paths_val_test, labels_train, labels_val_test = train_test_split(
    input_img_paths, breed_labels, train_size=train_frac,
    random_state=42, shuffle=True, stratify=breed_labels,
)
val_test_frac = 0.5
input_paths_val, input_paths_test, labels_val, labels_test = train_test_split(
    input_paths_val_test, labels_val_test, train_size=val_test_frac,
    random_state=42, shuffle=True, stratify=labels_val_test
)
del input_paths_val_test, labels_val_test
print(f"Train size ({train_frac}):", len(input_paths_train))
print(f"Val size ({round(val_test_frac * (1 - train_frac), 1)}):", len(input_paths_val))
print(f"Test size ({round(val_test_frac * (1 - train_frac), 1)}):", len(input_paths_test))

(PosixPath('images/n02090379-redbone'),
 PosixPath('images/n02085936-Maltese_dog'),
 PosixPath('images/n02088094-Afghan_hound'))

Résultat : un tuple de 639 images pour les 3 classes de races.
Distribution des races :
{'n02085936-Maltese_dog': 252,
 'n02088094-Afghan_hound': 239,
 'n02090379-redbone': 148}
Train size (0.8): 511
Val size (0.1): 64
Test size (0.1): 64


Classe personnalisée, compatible avec Keras, pour charger les images :

In [7]:
class ImageSequence(keras.utils.Sequence):
    def __init__(self, paths, labels, batch_size, transform=None, target_size=None):
        self.paths = paths
        self.labels = labels
        self.batch_size = batch_size
        self.transform = transform or transforms.Compose([
            transforms.Resize(target_size),  # Ensure consistent image dimensions
            transforms.ToTensor(),
        ])
        # Fix Error: "Invalid dtype: str704": Convert string labels to numerical values if needed
        if isinstance(self.labels[0], str):
            unique_labels = np.unique(self.labels)
            self.label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
            self.labels = np.array([self.label_to_idx[label] for label in self.labels])
        else:
            self.labels = np.array(self.labels)

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def __getitem__(self, idx):
        batch_paths = self.paths[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_labels = self.labels[idx*self.batch_size:(idx+1)*self.batch_size]

        batch_images = []
        for path in batch_paths:
            img = Image.open(path).convert('RGB')
            img = self.transform(img)
            # Convert from PyTorch NCHW to Keras NHWC format
            img_np = img.numpy()
            # PyTorch ToTensor() returns (C, H, W), Keras expects (H, W, C)
            if img_np.ndim == 3 and img_np.shape[0] == 3:  # Check if it's CHW format
                img_np = np.transpose(img_np, (1, 2, 0))  # CHW -> HWC
            elif img_np.ndim == 3 and img_np.shape[2] == 3:  # Already in HWC format
                pass  # No transformation needed
            else:
                raise ValueError(f"Unexpected image shape: {img_np.shape}")
            batch_images.append(img_np.astype(np.float32))

        return np.array(batch_images), np.array(batch_labels)

In [8]:
DEFAULT_TARGET_IMG_SIZE = (224, 224)
train_seq = ImageSequence(input_paths_train, labels_train, batch_size=32, target_size=DEFAULT_TARGET_IMG_SIZE)
val_seq = ImageSequence(input_paths_val, labels_val, batch_size=32, target_size=DEFAULT_TARGET_IMG_SIZE)
test_seq = ImageSequence(input_paths_test, labels_test, batch_size=32, target_size=DEFAULT_TARGET_IMG_SIZE)

#### a) Entraînement à 3 classes sans *data augmentation*

Construire le modèle :

In [9]:
IMG_SIZE   = 224
BATCH_SIZE = 64
EPOCHS     = 30
LR         = 1e-3
NUM_CLASSES = 120

data_augm = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomCrop(IMG_SIZE, IMG_SIZE),
    # layers.RandomRotation(0.1),   # ← décommenter selon besoin
    # layers.RandomZoom(0.1),
], name='data_augmentation')

def build_model(*, n_classes: int, include_data_augm: bool = True):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = data_augm(inputs) if include_data_augm else inputs
    x = layers.Rescaling(1./255)(x)

    # Bloc 1 ─ CHANGER : filtres, taille kernel, ajouter BatchNorm
    x = layers.Conv2D(32, 5, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)

    # Bloc 2 ─ CHANGER : ajouter d'autres blocs conv ici
    x = layers.Conv2D(64, 5, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)
    # x = layers.Dropout(0.5)(x)   # ← décommenter pour régulariser

    outputs = layers.Dense(n_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs)

model = build_model(n_classes=3, include_data_augm=False)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

Entraînement :

In [10]:
model.fit(train_seq, validation_data=val_seq, epochs=10)

Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 254ms/step - accuracy: 0.3659 - loss: 1.0830 - val_accuracy: 0.3906 - val_loss: 1.0746
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 248ms/step - accuracy: 0.3953 - loss: 1.0728 - val_accuracy: 0.3906 - val_loss: 1.0646
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 252ms/step - accuracy: 0.4364 - loss: 1.0524 - val_accuracy: 0.4844 - val_loss: 0.9694
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 265ms/step - accuracy: 0.5421 - loss: 0.9487 - val_accuracy: 0.4688 - val_loss: 0.9229
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 248ms/step - accuracy: 0.5753 - loss: 0.8778 - val_accuracy: 0.5156 - val_loss: 0.9021
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 249ms/step - accuracy: 0.5988 - loss: 0.8553 - val_accuracy: 0.5625 - val_loss: 0.8367
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 228ms/step - accuracy: 0.6595 - loss: 0.7891 - val_accuracy: 0.5469 - val_loss: 0.8727
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 254ms/step - accuracy: 0.6321 - loss: 0.7638 - val_accuracy: 0.

Évaluation :

In [11]:
model.evaluate(test_seq)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.5938 - loss: 0.8614


[0.861390233039856, 0.59375]

In [12]:
model.save('models/CNN_3_classes.keras', overwrite=True)

#### b) Entraînement à 3 classes avec *data augmentation*

Construire le modèle :

In [13]:
...

Ellipsis

Entraînement :

In [14]:
...

Ellipsis

Évaluation :

In [15]:
...

Ellipsis